<a href="https://colab.research.google.com/github/yoshiagent/IntControl-colab/blob/main/notebooks/Fuzzy_control.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 開啟"/></a>

# 模糊控制：原課程 Simulink 模型的 Python 版

原課程把模糊控制器（Fuzzy Logic Controller 方塊）放進 Simulink 模型，控制三個系統。這本筆記本把這些模型改寫成 Python 模擬，控制器直接讀原講義的 .fis 檔：

| 原講義附檔 | 系統 | 對照的網頁 |
|---|---|---|
| MCKFuzzy1.zip（FuzzyMCK1.mdl、MCKFZ.fis） | 質量–彈簧–阻尼（MCK），單輸入模糊控制器 | [MCK 模糊控制（一）](https://yoshiagent.github.io/IntControl/FZ-MCKFuzzy1.html) |
| MCKFuzzy2.zip、MCKFuzzy2Sugeno.zip、MCKFuzzySuL.zip | MCK，雙輸入 Mamdani／零階 Sugeno／一階 Sugeno | [MCK 模糊控制（二）](https://yoshiagent.github.io/IntControl/FZ-MCKFuzzy2.html) |
| sltank.mdl、tank.fis | 水槽液位 | [水位控制](https://yoshiagent.github.io/IntControl/FZ-TankLevelCon.html) |
| InvertCart.zip（slcp.mdl、slcp.fis） | 台車倒單擺 | [倒單擺 FIS](https://yoshiagent.github.io/IntControl/FZ-slcpFis.html) |

只用 Colab 內建的 numpy 與 plotly。模糊推論的算法和 MATLAB Fuzzy Logic Toolbox 相同（詳見另一本筆記本 Fuzzy_toolbox.ipynb）。

In [1]:
import io
import math
import re
import urllib.request
import zipfile
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

SITE = "https://yoshiagent.github.io/IntControl/files/2Controller/fuzzy/"

def fetch(name, member=None):
    """下載原講義的附檔；若是 zip，取出其中的 member 存到目前的資料夾"""
    with urllib.request.urlopen(SITE + name, timeout=30) as r:
        data = r.read()
    if member:
        data = zipfile.ZipFile(io.BytesIO(data)).read(member)
        name = member
    open(name, "wb").write(data)
    return name

for z, f in (("MCKFuzzy1.zip", "MCKFZ.fis"), ("MCKFuzzy2.zip", "MCKFZ2.fis"), ("MCKFuzzy2Sugeno.zip", "MCKFZ2sugeno.fis"),
             ("MCKFuzzySuL.zip", "MCKFZ2sugenoL.fis"), ("InvertCart.zip", "slcp.fis")):
    fetch(z, f)
fetch("tank.fis")

'tank.fis'

## 0. 模糊推論（和 Fuzzy_toolbox.ipynb 相同，只留控制需要的部分）

In [2]:
def mf(kind, p, x):
    x = np.asarray(x, float)
    if kind == "trimf":
        a, b, c = p
        l = (x - a) / (b - a) if b > a else (x >= b) * 1.0
        r = (c - x) / (c - b) if c > b else (x <= b) * 1.0
        return np.clip(np.minimum(l, r), 0, 1)
    if kind == "trapmf":
        a, b, c, d = p
        l = (x - a) / (b - a) if b > a else (x >= b) * 1.0
        r = (d - x) / (d - c) if d > c else (x <= c) * 1.0
        return np.clip(np.minimum(np.minimum(l, 1.0), r), 0, 1)
    if kind == "gaussmf":
        s, c = p
        return np.exp(-(x - c) ** 2 / (2 * s * s))
    if kind == "gbellmf":
        a, b, c = p[:3]
        return 1 / (1 + np.abs((x - c) / a) ** (2 * b))
    raise ValueError(kind)

def readfis(path):
    fis, sec, cur = dict(input=[], output=[], rule=[], type="mamdani", andMethod="min", orMethod="max", impMethod="min", aggMethod="max"), None, None
    for line in open(path, encoding="latin-1").read().splitlines():
        line = line.strip()
        if not line or line.startswith("%"):
            continue
        m = re.match(r"\[(\w+?)(\d*)\]", line)
        if m:
            sec = m.group(1)
            if sec in ("Input", "Output"):
                cur = dict(mf=[])
                fis["input" if sec == "Input" else "output"].append(cur)
            continue
        if sec == "Rules":
            a, rest = line.split(",")
            c, w, conn = re.match(r"\s*([-\d.\s]+)\(([\d.]+)\)\s*:\s*(\d)", rest).groups()
            fis["rule"].append(dict(antecedent=[int(float(v)) for v in a.split()], consequent=[int(float(v)) for v in c.split()], weight=float(w), connection=int(conn)))
            continue
        k, v = (s.strip() for s in line.split("=", 1))
        if sec == "System":
            fis[{"Name": "name", "Type": "type", "AndMethod": "andMethod", "OrMethod": "orMethod", "ImpMethod": "impMethod",
                 "AggMethod": "aggMethod", "DefuzzMethod": "defuzzMethod"}.get(k, k)] = v.strip("'")
        elif k == "Name":
            cur["name"] = v.strip("'")
        elif k == "Range":
            cur["range"] = [float(t) for t in v.strip("[]").split()]
        elif k.startswith("MF"):
            name, kind, p = re.match(r"'(.*?)':'(\w+)',\[(.*)\]", v).groups()
            cur["mf"].append(dict(name=name, type=kind, params=[float(t) for t in p.split()]))
    return fis

def evalfis(x, fis, npts=101):
    """一組輸入 → 輸出（Mamdani：依 FIS 的 And／Or／Imp／Agg 設定、重心法；Sugeno：加權平均）。輸入超出範圍時以邊界計算"""
    x = [min(max(v, var["range"][0]), var["range"][1]) for v, var in zip(x, fis["input"])]
    ws = []
    for r in fis["rule"]:
        degs = []
        for i, a in enumerate(r["antecedent"]):
            if a:
                m = fis["input"][i]["mf"][abs(a) - 1]
                d = float(mf(m["type"], m["params"], x[i]))
                degs.append(1 - d if a < 0 else d)
        if r["connection"] == 1:                                          # AND：min 或 prod
            w = min(degs) if fis["andMethod"] == "min" else float(np.prod(degs))
        else:                                                             # OR：max 或 probor
            w = max(degs) if fis["orMethod"] == "max" else 1 - float(np.prod([1 - d for d in degs]))
        ws.append(r["weight"] * w)
    ws = np.array(ws)
    out = fis["output"][0]
    lo, hi = out["range"]
    if fis["type"].lower() == "sugeno":
        zs = np.array([m["params"][0] if m["type"] == "constant" else float(np.dot(m["params"][:-1], x) + m["params"][-1])
                       for m in (out["mf"][abs(r["consequent"][0]) - 1] for r in fis["rule"])])
        return float(ws @ zs / ws.sum()) if ws.sum() > 0 else (lo + hi) / 2
    t = np.linspace(lo, hi, npts)
    agg = np.zeros(npts)
    for w, r in zip(ws, fis["rule"]):
        m = out["mf"][abs(r["consequent"][0]) - 1]
        y = mf(m["type"], m["params"], t)
        y = np.minimum(w, y) if fis["impMethod"] == "min" else w * y      # 截頂（min）或縮放（prod）
        agg = np.maximum(agg, y) if fis["aggMethod"] == "max" else agg + y
    return float(np.sum(agg * t) / np.sum(agg)) if agg.sum() > 0 else (lo + hi) / 2

def ds(a, k=5):
    """畫圖用：每 k 點取一點、取到小數 5 位（模擬本身仍用完整的步長），讓筆記本檔案小一點"""
    return np.round(np.asarray(a)[::k], 5)

def showrule(fis):
    names = lambda v, k: v["mf"][abs(k) - 1]["name"]
    for i, r in enumerate(fis["rule"]):
        cond = (" and " if r["connection"] == 1 else " or ").join(
            f"(in{j + 1} is {names(fis['input'][j], a)})" for j, a in enumerate(r["antecedent"]) if a)
        print(f"{i + 1}. If {cond} then (out is {names(fis['output'][0], r['consequent'][0])})")

## 1. MCK 系統的模糊控制

受控體是質量–彈簧–阻尼系統 m ẍ + c ẋ + k x = u（m = 1、c = 1、k = 2），目標是讓位移 x 追上單位步階。Simulink 模型的結構：

* **單輸入（FuzzyMCK1.mdl）**：步階（t = 1 s 由 0 到 1）→ 誤差 e = r − x → 模糊控制器 → u → MCK。
* **雙輸入（FuzzyMCK2*.mdl）**：步階（t = 0）→ e 與 de/dt（Derivative 方塊）→ Mux → 模糊控制器 → u → MCK。設定值固定時 de/dt = −ẋ；Simulink 的 Derivative 方塊在 t = 0 輸出 0，這裡也一樣（x、ẋ 的初值都是 0）。

用四階 Runge–Kutta、步長 0.004 s 積分，每個 RK 子步都重新算一次模糊控制器（和 Simulink 的連續積分相同）。

In [3]:
M_, C_, K_ = 1.0, 1.0, 2.0

def mck(fis, two_inputs, t_end=10.0, dt=0.004, t_step=0.0):
    ranges = [v["range"] for v in fis["input"]]
    x = v = 0.0
    ts, xs, us = [], [], []
    for i in range(int(round(t_end / dt)) + 1):
        t = i * dt
        r = 1.0 if t >= t_step else 0.0
        def acc(x_, v_):
            inp = (r - x_, -v_) if two_inputs else (r - x_,)
            u = evalfis([min(max(a, lo), hi) for a, (lo, hi) in zip(inp, ranges)], fis)
            return (u - C_ * v_ - K_ * x_) / M_, u
        a1, u = acc(x, v)
        ts.append(t); xs.append(x); us.append(u)
        a2, _ = acc(x + dt / 2 * v, v + dt / 2 * a1)
        a3, _ = acc(x + dt / 2 * (v + dt / 2 * a1), v + dt / 2 * a2)
        a4, _ = acc(x + dt * (v + dt / 2 * a2), v + dt * a3)
        x, v = x + dt / 6 * (v + 2 * (v + dt / 2 * a1) + 2 * (v + dt / 2 * a2) + (v + dt * a3)), v + dt / 6 * (a1 + 2 * a2 + 2 * a3 + a4)
    return np.array(ts), np.array(xs), np.array(us)

def steady_state(fis, two_inputs):
    """穩態：ẋ = 0 時 u = F(e, 0) 必須等於彈簧力 k·x = k(1 − e)，在細格點上找交點"""
    es = np.linspace(-0.6, 0.6, 12001)
    g = np.array([evalfis([e, 0] if two_inputs else [e], fis) for e in es]) - K_ * (1 - es)
    i = int(np.where(np.sign(g[:-1]) != np.sign(g[1:]))[0][0])
    return 1 - es[i]

FIS = {"單輸入 Mamdani（MCKFZ.fis）": ("MCKFZ.fis", False, 1.0), "雙輸入 Mamdani（MCKFZ2.fis）": ("MCKFZ2.fis", True, 0.0),
       "雙輸入 零階 Sugeno（MCKFZ2sugeno.fis）": ("MCKFZ2sugeno.fis", True, 0.0), "雙輸入 一階 Sugeno（MCKFZ2sugenoL.fis）": ("MCKFZ2sugenoL.fis", True, 0.0)}
for name, (f, two, _) in FIS.items():
    print(f"=== {name}")
    showrule(readfis(f))

=== 單輸入 Mamdani（MCKFZ.fis）
1. If (in1 is Negative) then (out is NF)
2. If (in1 is Zero) then (out is ZF)
3. If (in1 is Positive) then (out is PF)
=== 雙輸入 Mamdani（MCKFZ2.fis）
1. If (in1 is P) and (in2 is P) then (out is PL)
2. If (in1 is P) and (in2 is N) then (out is PM)
3. If (in1 is ZR) and (in2 is P) then (out is PS)
4. If (in1 is ZR) and (in2 is N) then (out is NS)
5. If (in1 is N) and (in2 is P) then (out is NM)
6. If (in1 is N) and (in2 is N) then (out is NL)
=== 雙輸入 零階 Sugeno（MCKFZ2sugeno.fis）
1. If (in1 is P) and (in2 is P) then (out is PL)
2. If (in1 is P) and (in2 is N) then (out is PM)
3. If (in1 is Z) and (in2 is P) then (out is PS)
4. If (in1 is Z) and (in2 is N) then (out is NS)
5. If (in1 is N) and (in2 is P) then (out is NM)
6. If (in1 is N) and (in2 is N) then (out is NL)
=== 雙輸入 一階 Sugeno（MCKFZ2sugenoL.fis）
1. If (in1 is P) and (in2 is P) then (out is PL)
2. If (in1 is P) and (in2 is N) then (out is PM)
3. If (in1 is Z) and (in2 is P) then (out is PS)
4. If (in1 is Z)

In [4]:
fig = go.Figure()
results = {}
for name, (f, two, ts_) in FIS.items():
    fis = readfis(f)
    t, x, u = mck(fis, two, t_step=ts_)
    xss = steady_state(fis, two)
    results[name] = (t, x, u, xss)
    print(f"{name}：峰值 {x.max():.3f}，穩態 {xss:.4f}")
    fig.add_scatter(x=ds(t), y=ds(x), mode="lines", name=name.split("（")[0])
fig.add_hline(y=1, line_dash="dash", line_color="gray")
fig.update_layout(title="MCK system with fuzzy controllers (step input)", xaxis_title="t (s)", yaxis_title="x", height=400)
fig.show()

單輸入 Mamdani（MCKFZ.fis）：峰值 1.330，穩態 0.9332


雙輸入 Mamdani（MCKFZ2.fis）：峰值 1.126，穩態 1.0158


雙輸入 零階 Sugeno（MCKFZ2sugeno.fis）：峰值 1.072，穩態 1.0755


雙輸入 一階 Sugeno（MCKFZ2sugenoL.fis）：峰值 1.065，穩態 1.0702


四個控制器都沒有積分作用，所以穩態不一定剛好是 1：穩態時 ẋ = 0、控制力 F(e, 0) 必須剛好等於彈簧力 k·x。

* 單輸入：穩態 0.933，有約 7% 的穩態誤差，而且峰值約 1.33（只看誤差、沒有阻尼）。
* 雙輸入 Mamdani：e = ė = 0 時只有「零 且 正」這條規則被觸發，輸出是 3（正小的重心），大於維持 x = 1 所需的 2，所以穩態略超過 1（1.016）。
* 零階、一階 Sugeno：穩態 1.075、1.070，響應單調上升、沒有超越。

詳細的分析與規則表見 [MCK 模糊控制（二）](https://yoshiagent.github.io/IntControl/FZ-MCKFuzzy2.html)。

In [5]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("position x", "controller output u"))
for name, (t, x, u, _) in results.items():
    nm = name.split("（")[0]
    fig.add_scatter(x=ds(t), y=ds(x), mode="lines", name=nm, legendgroup=nm, row=1, col=1)
    fig.add_scatter(x=ds(t), y=ds(u), mode="lines", name=nm, legendgroup=nm, showlegend=False, row=2, col=1)
fig.update_xaxes(title_text="t (s)", row=2, col=1); fig.update_layout(height=560)
fig.show()

## 2. 水槽液位控制（sltank.mdl、tank.fis）

依 sltank.mdl 的參數：槽高 2 m、底面積 1 m²、出水管 0.05 m²、初始水位 0.5 m。進水量 = 0.5 × 閥開度（0～1），出水量 = 0.05 √(2 g h)。
設定值是 1 ± 0.5 m 的方波（頻率 0.1 rad/s）。tank.fis 的輸入是水位誤差（level，限制在 ±1）與水位變化率（rate，限制在 ±0.1），輸出是**閥的開關速度**（valve），積分後才是閥開度。
用固定步長 0.05 s 模擬 200 s：

In [6]:
tank_fis = readfis("tank.fis")

def tank(t_end=200.0, dt=0.05, fis=tank_fis, setpoint=None):
    h, valve = 0.5, 0.0
    out = []
    for i in range(int(round(t_end / dt)) + 1):
        t = i * dt
        r = setpoint if setpoint is not None else 1.0 + 0.5 * (1 if math.sin(0.1 * t) >= 0 else -1)
        dh = (0.5 * valve - 0.05 * math.sqrt(2 * 9.8 * max(h, 0))) / 1.0
        u = evalfis([max(-1, min(1, r - h)), max(-0.1, min(0.1, dh))], fis)
        out.append((t, h, r, valve))
        valve = min(1.0, max(0.0, valve + u * dt))
        h = min(2.0, max(0.0, h + dh * dt))
    return np.array(out)

o = tank()
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("water level", "valve opening"))
fig.add_scatter(x=ds(o[:, 0], 4), y=ds(o[:, 2], 4), mode="lines", line=dict(dash="dash", color="gray"), name="set point", row=1, col=1)
fig.add_scatter(x=ds(o[:, 0], 4), y=ds(o[:, 1], 4), mode="lines", name="level h", row=1, col=1)
fig.add_scatter(x=ds(o[:, 0], 4), y=ds(o[:, 3], 4), mode="lines", name="valve", row=2, col=1)
fig.update_xaxes(title_text="t (s)", row=2, col=1); fig.update_layout(height=520)
fig.show()

水位都能跟上設定值，但會留一點點誤差。原因在 tank.fis 的輸出歸屬函數不對稱：誤差 0、變化率 0 時輸出不是 0（是 −0.00076），控制器要在誤差約 +1 cm 的地方才會停止開關閥。
把「慢關」改成和「慢開」對稱的 [−0.4 −0.3 −0.2]，平衡誤差就變成 0。固定設定值 1.5 m 跑 300 s 比較：

In [7]:
print("誤差 0、變化率 0 時的輸出：", round(evalfis([0, 0], tank_fis), 5))
import copy
tank_sym = copy.deepcopy(tank_fis)
print("原本的慢關：", tank_sym["output"][0]["mf"][1])
tank_sym["output"][0]["mf"][1]["params"] = [-0.4, -0.3, -0.2]
for label, f in (("原本的 tank.fis", tank_fis), ("慢關改成對稱", tank_sym)):
    oo = tank(t_end=300, fis=f, setpoint=1.5)
    print(f"{label}：300 s 後水位 {oo[-1, 1]:.4f} m（誤差 {round((1.5 - oo[-1, 1]) * 100, 2) + 0.0:+.2f} cm）")

誤差 0、變化率 0 時的輸出： -0.00076
原本的慢關： {'name': 'close_slow', 'type': 'trimf', 'params': [-0.6, -0.5, -0.4]}


原本的 tank.fis：300 s 後水位 1.4900 m（誤差 +1.00 cm）
慢關改成對稱：300 s 後水位 1.5000 m（誤差 +0.00 cm）


## 3. 台車倒單擺（InvertCart.zip：slcp.mdl、slcp.fis）

擺桿長 1 m（質心到鉸鏈 l = 0.5 m）、質量 m = 0.1 kg，台車 M = 1 kg，g = 9.8 m/s²。θ 以直立為 0、向 +x 傾為正：

$$\ddot\theta = \frac{g\sin\theta + \cos\theta\,\dfrac{-F - m l \dot\theta^2 \sin\theta}{M + m}}{l\left(\dfrac43 - \dfrac{m\cos^2\theta}{M + m}\right)},\qquad
\ddot x = \frac{F + m l(\dot\theta^2\sin\theta - \ddot\theta\cos\theta)}{M + m}$$

slcp.fis 是 Sugeno 型，四個輸入 [θ, θ̇, 台車位置 − 目標, 台車速度] 各 2 個 gbellmf、16 條規則，輸出是線性函數（推力 F）。目標位置是 ±1.5 m 的方波（週期 10 s）。
控制器每 0.01 s 算一次推力，台車用 RK4 積分：

In [8]:
cp_fis = readfis("slcp.fis")
P = dict(l=0.5, m=0.1, mc=1.0, g=9.8)

def cart_acc(th, thd, F):
    s, c = math.sin(th), math.cos(th)
    tot = P["mc"] + P["m"]
    thdd = (P["g"] * s + c * ((-F - P["m"] * P["l"] * thd * thd * s) / tot)) / (P["l"] * (4 / 3 - P["m"] * c * c / tot))
    return thdd, (F + P["m"] * P["l"] * (thd * thd * s - thdd * c)) / tot

def cartpole(t_end=20.0, dt=0.01):
    th = thd = x = xd = 0.0
    out = []
    ranges = [v["range"] for v in cp_fis["input"]]
    for i in range(int(round(t_end / dt)) + 1):
        t = i * dt
        tgt = 1.5 * (1 if math.sin(0.62832 * t) >= 0 else -1)
        F = evalfis([min(max(a, lo), hi) for a, (lo, hi) in zip([th, thd, x - tgt, xd], ranges)], cp_fis)
        out.append((t, th, x, tgt, F))
        f = lambda s: np.array([s[1], cart_acc(s[0], s[1], F)[0], s[3], cart_acc(s[0], s[1], F)[1]])
        st = np.array([th, thd, x, xd])
        k1 = f(st); k2 = f(st + dt / 2 * k1); k3 = f(st + dt / 2 * k2); k4 = f(st + dt * k3)
        th, thd, x, xd = st + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    return np.array(out)

cp = cartpole()
print(f"20 s 內最大擺角 {np.abs(cp[:, 1]).max():.3f} rad（{math.degrees(np.abs(cp[:, 1]).max()):.1f}°）")
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=("cart position", "pole angle (rad)", "force F (N)"))
fig.add_scatter(x=ds(cp[:, 0], 2), y=ds(cp[:, 3], 2), mode="lines", line=dict(dash="dash", color="gray"), name="target", row=1, col=1)
fig.add_scatter(x=ds(cp[:, 0], 2), y=ds(cp[:, 2], 2), mode="lines", name="x", row=1, col=1)
fig.add_scatter(x=ds(cp[:, 0], 2), y=ds(cp[:, 1], 2), mode="lines", name="theta", row=2, col=1)
fig.add_scatter(x=ds(cp[:, 0], 2), y=ds(cp[:, 4], 2), mode="lines", name="F", row=3, col=1)
fig.update_xaxes(title_text="t (s)", row=3, col=1); fig.update_layout(height=640)
fig.show()

20 s 內最大擺角 0.273 rad（15.6°）


換目標時，台車會先往**反方向**退一點，讓擺桿往目標方向傾斜，再加速追上去，這和人用手平衡竹竿的做法一樣。

### 動畫

原講義用 S-function `animcp.m`（MathWorks 的範例程式，這裡不照抄）在模擬時畫台車與擺桿。這裡用 plotly 畫一個同樣內容的動畫：台車、擺桿、推力（橘色，長度和推力成正比）與目標位置（紅色三角形）。按 ▶ 播放（每 0.2 s 一張）：

In [9]:
def cart_xy(k):
    """第 k 步的台車、擺桿、推力箭頭、目標位置（只回傳座標，動畫的每一張只更新座標）"""
    t, th, x, tgt, F = cp[k]
    w, hgt = 0.4, 0.2
    r = lambda v: [round(float(a), 3) for a in v]
    s = 1 if F >= 0 else -1
    tail = x - s * (w / 2)                                   # 推力畫在台車側面，長度 ∝ |F|
    return [dict(x=r([x - w / 2, x + w / 2, x + w / 2, x - w / 2, x - w / 2]), y=[0, 0, hgt, hgt, 0]),
            dict(x=r([x, x + math.sin(th)]), y=r([hgt, hgt + math.cos(th)])),
            dict(x=r([tail - s * min(abs(F), 20) / 20, tail]), y=[hgt / 2, hgt / 2]),
            dict(x=[round(float(tgt), 3)], y=[-0.08]),
            dict(x=[-2.6], y=[1.35], text=[f"t = {t:.1f} s"])]

base = cart_xy(0)
styles = [dict(fill="toself", mode="lines", line=dict(color="royalblue"), fillcolor="lightsteelblue"),
          dict(mode="lines+markers", line=dict(color="black", width=5), marker=dict(size=[4, 10])),
          dict(mode="lines", line=dict(color="orange", width=6)),
          dict(mode="markers", marker=dict(symbol="triangle-up", size=14, color="red")),
          dict(mode="text", textposition="middle right")]
idx = list(range(0, len(cp), 20))                            # 每 0.2 s 一張
fig = go.Figure(data=[go.Scatter(**b, **s) for b, s in zip(base, styles)],
                frames=[go.Frame(data=[go.Scatter(**d) for d in cart_xy(k)], name=str(k)) for k in idx])
fig.add_shape(type="line", x0=-3, x1=3, y0=0, y1=0, line=dict(color="gray"))
fig.update_layout(
    title="Cart & Pole (fuzzy controller slcp.fis)", showlegend=False, width=680, height=380,
    xaxis=dict(range=[-3, 3], title="x (m)"), yaxis=dict(range=[-0.2, 1.45], scaleanchor="x", visible=False),
    updatemenus=[dict(type="buttons", x=0, y=-0.15, xanchor="left", buttons=[
        dict(label="▶ play", method="animate", args=[None, dict(frame=dict(duration=200, redraw=False), transition=dict(duration=0), fromcurrent=True)]),
        dict(label="❚❚ pause", method="animate", args=[[None], dict(mode="immediate", frame=dict(duration=0))])])],
    sliders=[dict(active=0, x=0.2, len=0.8, y=-0.1, currentvalue=dict(prefix="frame "),
                  steps=[dict(label=f"{cp[k, 0]:.1f}", method="animate", args=[[str(k)], dict(mode="immediate", frame=dict(duration=0, redraw=False))]) for k in idx])])
fig.show()

## 4. 把 FIS 放進 Simulink（fis2sim.mdl）

[Working With Simulink](https://yoshiagent.github.io/IntControl/FZ-fis2simFlash.html) 一頁的 fis2sim.mdl 只有一個 Fuzzy Logic Controller 方塊，參數是工作區中 FIS 變數的名稱（`FISMatrix`）。
Simulink 每個積分步都把方塊的輸入送進 FIS、取得輸出。在 Python 裡，這就是在模擬迴圈中每一步呼叫一次 `evalfis`，上面三個例子都是這樣寫的：

```python
for each time step:
    inputs = [error, error_rate, ...]      # 方塊的輸入（Mux）
    u = evalfis(inputs, fis)                # Fuzzy Logic Controller 方塊
    state = integrate(plant, state, u, dt)  # 受控體
```

網站上的 [MCK](https://yoshiagent.github.io/IntControl/FZ-MCKFuzzy2.html)、[水槽](https://yoshiagent.github.io/IntControl/FZ-TankModel.html)、
[倒單擺](https://yoshiagent.github.io/IntControl/FZ-slcpFis.html)各頁也有可以即時操作的互動模擬。